# 반 걸음 — T4 실제 AI 실행
- 모델: Kanana 2 3B / BGE-M3
- 입력: bangeol-real-ai.zip (수정한 화면·활동 목록·실제 서비스 5,288건)
- 출력: 실제 모델 검증 결과, Streamlit 미리보기
- 런타임 → 런타임 유형 변경 → T4 GPU 선택 후 위에서부터 실행
- 모델 다운로드와 첫 서비스 임베딩은 수 분 이상 소요될 수 있음
- 실행 중인 노트북 안에서 모델 검증·화면 확인용으로 사용


In [ ]:
import subprocess, sys, os, time, json, hashlib, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "T4 GPU를 선택한 뒤 다시 실행해 주세요."
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)
os.environ["BANGEOL_EMBED_BATCH_SIZE"] = "4"
os.environ["BANGEOL_PIPELINE_MODE"] = "실제 모델"

import os
from google.colab import userdata
os.environ["BANGEOL_REQUIRE_REVIEW_DATA"] = "1"
try:
    _key = userdata.get("GEMINI_API_KEY")
except Exception:
    _key = None
if _key:
    os.environ["GEMINI_API_KEY"] = _key
    print("Gemini 키: 있음")
else:
    os.environ.pop("GEMINI_API_KEY", None)
    print("Gemini 키: 없음 (일상 대화는 기본 응답 사용)")
del _key


## 1. 최신 파일 묶음 업로드
- 아래 셀 실행 후 `bangeol-real-ai.zip` 선택
- GitHub 토큰 입력 없이 현재 수정본을 그대로 실행


In [ ]:
from google.colab import files
uploaded = files.upload()
assert "bangeol-real-ai.zip" in uploaded, "bangeol-real-ai.zip을 선택해 주세요."
ROOT = Path("/content/bangeol-real-ai")
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile("bangeol-real-ai.zip") as z:
    for member in z.infolist():
        target = (ROOT / member.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), "잘못된 압축 경로"
    z.extractall(ROOT)
manifest = json.loads((ROOT / "bundle-manifest.json").read_text(encoding="utf-8"))
for name, digest in manifest["files"].items():
    assert hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest, name
print("최신 파일 검증:", len(manifest["files"]), "개")
del uploaded

master_hash = hashlib.sha256((ROOT / "data/processed/services_master.csv").read_bytes()).hexdigest()
for relative, schema in [("outputs/03_q1_recommendation/q1_review.json", "q1-review-v1"),
                         ("outputs/04_q2_recommendation/q2_info.json", "q2-info-v1")]:
    data = json.loads((ROOT / relative).read_text(encoding="utf-8"))
    assert data["schema"] == schema and data["master_sha256"] == master_hash
    print(schema, len(data["services"]), "건 연결")
print("로컬 수정 포함:", manifest.get("working_tree_dirty"))


## 2. 실행 패키지 설치
- 팀 requirements 기준 설치
- 화면은 로컬 검증과 같은 Streamlit 1.67.0 사용
- Transformers는 4.57 계열 호환 범위 사용


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "ai/requirements-streamlit.txt")], check=True)


## 3. 실제 모델 사전 검증 · 최초 인수 시 권장
- 이전 전체 설명 생성 방식의 모델 검증 실행
- 전체 설명 5개 생성으로 수 분 소요 가능
- 파일 설치가 유지된 상태에서 화면만 재실행할 때는 건너뛰기 가능
- 별도 프로세스 종료로 화면과 모델 이중 적재 방지


In [ ]:
with (ROOT / "real-model-check.log").open("w", encoding="utf-8") as log:
    check = subprocess.Popen([sys.executable, "-u", str(ROOT / "ai/scripts/real_model_check.py")],
                             cwd=ROOT / "ai", env=os.environ.copy(), stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in check.stdout:
        print(line, end="")
        log.write(line)
    result = check.wait()
assert result == 0, "실제 모델 검증 실패: 위 오류를 확인해 주세요. 가짜 모델로 전환하지 않습니다."


## 4. 실제 AI 화면 실행
- 상단 배지 `실제 Pipeline 연결` 확인
- 결과는 서비스 목록과 원문 소개부터 표시
- AI 설명은 상세에서 요청 시 생성
- 빈 화면이면 이 셀 재실행 후 새 탭 링크 사용
- 런타임 초기화로 앱 폴더가 없으면 ZIP 업로드·설치부터 복원


In [ ]:
import runpy
launch = runpy.run_path(str(ROOT / "ai/scripts/colab_launch.py"))
streamlit_process = launch["start_preview"](ROOT, globals().get("streamlit_process"))


## 5. 실행 상태 확인
- 모델 내려받기·오류는 아래 로그로 확인
- 검증 결과와 GPU 수치만 공유, 실제 사용자 대화는 공유하지 않음


In [ ]:
print((ROOT / "streamlit.log").read_text(encoding="utf-8")[-10000:])
subprocess.run(["nvidia-smi"], check=True)
